In [ ]:
"""
FePt Hemispherical Cap — Hysteresis Loop Simulation
====================================================
Simulates hysteresis loops for FePt hemispherical caps using OOMMF via the
ubermag Python interface. Caps consist of a mixed L10/A1 phase with radial
uniaxial anisotropy. The SiO2 sphere is NOT included — only the FePt shell
is modelled.

Solver is chosen automatically based on the temperature input at runtime:
  - T = 0 K  →  MinDriver (static energy minimisation, no dynamics)
  - T > 0 K  →  TimeDriver (LLG time integration with stochastic thermal field)

Inputs  : Physical parameters defined in Section 1 (diameters, Ms, A, Ku,
          soft phase fraction, field range) + temperature entered at runtime.
Outputs : Per-cap CSV files and a combined master CSV + SVG summary plot,
          saved to a timestamped output folder.
"""

# --- Standard library ---
import os
from datetime import datetime

# --- Third-party ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Micromagnetics stack ---
import micromagneticmodel as mm
import discretisedfield as df
import oommfc as oc


# ===========================================================================
# 1. PARAMETERS & STUDY SETUP
# ===========================================================================

# --- Temperature selection (determines solver) ---
T = int(input(
    "Enter simulation temperature in Kelvin (0 = static MinDriver, >0 = dynamic TimeDriver): "
))
USE_DYNAMIC = T > 0  # True → TimeDriver + LLG dynamics; False → MinDriver

# --- LLG dynamics parameters (only used when T > 0) ---
alpha    = 0.05   # Gilbert damping constant (dimensionless)
run_time = 1e-10  # Integration time per field step [s]

# Soft (A1) phase fraction — controls how many cells get Ku_soft vs Ku_hard
SOFT_FRACTION_A1 = 0.50

# Sphere diameters to simulate [m]
diameters = [1e-6, 3e-6]

# --- Material parameters (L10 FePt literature values) ---
Ms      = 1.0e6   # Saturation magnetisation [A/m]
A       = 1.0e-11 # Exchange stiffness [J/m]
Ku_hard = 6.6e6   # Uniaxial anisotropy constant — L10 (hard) phase [J/m^3]
Ku_soft = 1.0e4   # Uniaxial anisotropy constant — A1 (soft) phase [J/m^3]

# --- Field sweep parameters ---
mu0    = 4 * np.pi * 1e-7  # Vacuum permeability [H/m]
B_max  = 18.0               # Maximum applied field magnitude [T]
B_tilt = 0.02               # Small in-plane tilt to break symmetry and avoid saddle points [T]

# --- Geometry ---
cap_thickness = 60e-9  # FePt cap thickness [m]

# --- Anisotropy mode (Uniaxial_Vertical kept for reference but disabled) ---
anisotropy_modes = [
    "Radial",
    # "Uniaxial_Vertical"  # Disabled: focus is on radial (axial) symmetry
]

# --- Reproducibility & output ---
np.random.seed(42)  # Fix random seed for reproducible A1/L10 phase assignment
soft_pct  = int(SOFT_FRACTION_A1 * 100)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

# Folder name includes temperature so static and dynamic runs never collide
T_label       = f"T{T}K" if USE_DYNAMIC else "0K_static"
OUTPUT_FOLDER = f'Janus_Study_{soft_pct}pct_A1_{T_label}_{timestamp}'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Full field sweep: high → low → high (242 steps total)
B_fields = np.concatenate([
    np.linspace(B_max, -B_max, 121),
    np.linspace(-B_max, B_max, 121)
])

successful_runs = {}  # Collects {run_name: Mz/Ms array} for the final export

solver_label = f"TimeDriver @ {T} K" if USE_DYNAMIC else "MinDriver (0 K)"
print(f"\n--- MASTER STUDY: {soft_pct}% A1 Phase | Radial Mode | {solver_label} ---")
print(f"Output Directory: {OUTPUT_FOLDER}\n")


# ===========================================================================
# 2. MAIN EXECUTION LOOP
# ===========================================================================

for d_idx, sphere_diameter in enumerate(diameters):
    for mode in anisotropy_modes:

        # Build a filesystem-safe run identifier
        size_um   = sphere_diameter * 1e6
        safe_name = f"cap_{str(size_um).replace('.', '_')}um_{mode}_{soft_pct}pct_A1"
        checkpoint_path = os.path.join(OUTPUT_FOLDER, f"data_{safe_name}_{timestamp}.csv")

        # --- Resume support: skip if this run already has saved data ---
        if os.path.exists(checkpoint_path):
            print(f">> Skipping {safe_name}, data found.")
            existing_data = pd.read_csv(checkpoint_path)
            successful_runs[safe_name] = existing_data['Mz/Ms'].values
            continue

        # --- Geometry: hemispherical shell radii ---
        R_inner = sphere_diameter / 2
        R_outer = R_inner + cap_thickness

        # Mesh resolution: ~18 nm cells for small caps, fixed 110 cells for larger ones
        # to keep computation tractable while resolving the exchange length (~5 nm)
        n_val = int(round((2 * R_outer) / 18e-9)) if sphere_diameter <= 1.5e-6 else 110

        # Upper half-space only (z >= 0) — models just the cap, not the full sphere
        region = df.Region(p1=(-R_outer, -R_outer, 0),
                           p2=( R_outer,  R_outer, R_outer))
        mesh = df.Mesh(region=region, n=(n_val, n_val, n_val))

        print(f"\n[RUNNING {d_idx+1}/{len(diameters)}] {safe_name} | Mesh: {n_val}^3")

        # --- Spatial field functions assigned cell-by-cell ---

        def Ms_mask(pos):
            """Return Ms inside the hemispherical shell, 0 elsewhere."""
            r = np.linalg.norm(pos)
            return Ms if (R_inner <= r <= R_outer and pos[2] >= 0) else 0

        def Ku_distribution(pos):
            """Randomly assign L10 (hard) or A1 (soft) Ku based on SOFT_FRACTION_A1."""
            r = np.linalg.norm(pos)
            if R_inner <= r <= R_outer and pos[2] >= 0:
                return Ku_soft if np.random.random() < SOFT_FRACTION_A1 else Ku_hard
            return 0

        def u_func(pos):
            """
            Easy-axis direction per cell.
            Radial mode: axis points along the local surface normal (outward radial).
            Falls back to (0,0,1) at the origin to avoid division by zero.
            """
            if mode == "Radial":
                r = np.linalg.norm(pos)
                return (pos[0]/r, pos[1]/r, pos[2]/r) if r != 0 else (0, 0, 1)
            return (0, 0, 1)  # Placeholder for Uniaxial_Vertical (currently disabled)

        # --- Build micromagnetic system ---
        system = mm.System(name=safe_name)
        system.energy = (
            mm.Exchange(A=A)
            + mm.Demag()
            + mm.UniaxialAnisotropy(
                K=df.Field(mesh, nvdim=1, value=Ku_distribution),
                u=df.Field(mesh, nvdim=3, value=u_func)
            )
            + mm.Zeeman(H=(0, 0, 0))  # Field updated at each step below
        )

        # LLG dynamics terms — required by TimeDriver, not used by MinDriver.
        # Precession drives magnetisation around the effective field;
        # Damping dissipates energy and allows convergence toward equilibrium.
        if USE_DYNAMIC:
            system.dynamics = (
                mm.Precession(gamma0=mm.consts.gamma0)
                + mm.Damping(alpha=alpha)
            )

        # Initialise magnetisation uniformly along +z
        system.m = df.Field(mesh, nvdim=3, value=(0, 0, 1),
                            norm=df.Field(mesh, nvdim=1, value=Ms_mask))

        # --- Volume sanity check: compare analytical vs discretised shell volume ---
        # Discrepancy arises from the staircase approximation of the curved boundary
        v_theory = (2/3) * np.pi * (R_outer**3 - R_inner**3)
        v_mesh   = (system.m.norm.integrate() / Ms).item()
        print(f"  Vol Error: {abs(v_mesh - v_theory)/v_theory*100:.2f}%")

        # --- Field sweep ---
        hyst_data = []

        if USE_DYNAMIC:
            # TimeDriver integrates the LLG equation over run_time per step;
            # T=T activates OOMMF's stochastic thermal field (Langevin dynamics)
            driver = oc.TimeDriver()
        else:
            # MinDriver minimises total energy with no dynamics — equivalent to 0 K
            driver = oc.MinDriver()

        for s_idx, B_z in enumerate(B_fields):
            try:
                # Small By tilt prevents the solver getting stuck at a saddle point
                system.energy.zeeman.H = (0, B_tilt / mu0, B_z / mu0)

                if USE_DYNAMIC:
                    driver.drive(system, t=run_time, n=1, T=T, n_threads=8)
                else:
                    driver.drive(system, n_threads=8,
                                 stopping_mxHxm=2e4,         # Convergence criterion
                                 total_iteration_limit=15000) # Safety cap on iterations

                # Normalised average Mz over magnetic material only
                m_norm_int = system.m.norm.integrate().item()
                mz_material_avg = (
                    system.m.z.integrate().item() / m_norm_int
                ) if m_norm_int > 0 else 0
                hyst_data.append(mz_material_avg)

                # Progress log every 40 steps
                if s_idx % 40 == 0:
                    print(f"   Step {s_idx+1}/242 | B: {B_z:6.2f} T | Mz/Ms: {mz_material_avg:8.4f}")

            except Exception as e:
                # On OOMMF crash: carry forward last known value to maintain array length
                fallback = hyst_data[-1] if hyst_data else 0
                hyst_data.append(fallback)

        # --- Save per-cap results ---
        if len(hyst_data) == len(B_fields):
            df_temp = pd.DataFrame({'B_ext (T)': B_fields, 'Mz/Ms': hyst_data})
            df_temp.to_csv(checkpoint_path, index=False)
            successful_runs[safe_name] = hyst_data
            print(f"  >> Saved: {checkpoint_path}")
        else:
            print(f"  !! ERROR: Length mismatch for {safe_name}. Check for OOMMF crashes.")

        oc.delete(system)  # Free OOMMF resources before next run


# ===========================================================================
# 3. FINAL SUMMARY & EXPORTS
# ===========================================================================

if successful_runs:
    master_results = pd.DataFrame({'B_ext (T)': B_fields})
    plt.figure(figsize=(12, 8))

    for key, data in successful_runs.items():
        master_results[key] = data
        plt.plot(B_fields, data, label=key.replace('_A1', ''))

    # Save combined CSV with all diameters / modes as columns
    master_csv_name = f"MASTER_DATA_{soft_pct}pct_A1_{T_label}_{timestamp}.csv"
    master_results.to_csv(os.path.join(OUTPUT_FOLDER, master_csv_name), index=False)

    plt.xlabel('B_ext (T)')
    plt.ylabel('Mz/Ms (Material Normalised)')
    plt.grid(True)
    plt.legend(fontsize='x-small', ncol=2)
    plt.title(f'FePt Radial Study: {soft_pct}% A1 | {solver_label} ({timestamp})')

    plot_name = f'SUMMARY_PLOT_{soft_pct}pct_A1_{T_label}_{timestamp}.svg'
    plt.savefig(os.path.join(OUTPUT_FOLDER, plot_name))
    plt.show()

print(f"\nAll tasks complete. Files saved in: {OUTPUT_FOLDER}")
